# 03 · Tool Design & the Tool Registry

Good tools make good agents. A `ToolRegistry` lets you register your own Python functions with a JSON schema, hand all schemas to the model, and **dispatch calls safely** — errors and timeouts become observations instead of crashes. Independent calls can even run **in parallel**.

> Runs entirely on Ollama.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'agents_lab').exists())
sys.path.insert(0, str(root))

Register a custom tool with the decorator form. The description + schema are what the model sees, so make them clear.

In [ ]:
from agents_lab import ToolRegistry

registry = ToolRegistry()

@registry.register(
    description='Return the length of a word.',
    parameters={'type': 'object', 'properties': {'word': {'type': 'string'}}, 'required': ['word']},
)
def word_length(word):
    return len(word)

print('registered:', registry.names)
print('schema[0]:', registry.schemas[0])

**Safe dispatch:** a bad tool call comes back as an error string the agent can read and recover from — it never raises.

In [ ]:
print(registry.dispatch('word_length', {'word': 'agent'}))       # 5
print(registry.dispatch('word_length', {'wrong': 'x'}))          # error -> text
print(registry.dispatch('missing_tool', {}))                     # unknown -> text

**Timeouts** protect against a hanging tool. Give a slow tool a short `timeout_s` and it fails fast.

In [ ]:
import time
from agents_lab import Tool

registry.add(Tool(
    name='slow', description='A slow tool.',
    func=lambda: (time.sleep(2), 'done')[1],
    parameters={'type': 'object', 'properties': {}}, timeout_s=0.5,
))
print(registry.dispatch('slow', {}))

**Parallel calls:** independent tool calls run concurrently, preserving order.

In [ ]:
calls = [
    {'name': 'word_length', 'arguments': {'word': 'multi'}},
    {'name': 'word_length', 'arguments': {'word': 'agent'}},
]
print(registry.run_parallel(calls))

**Takeaway:** a registry gives you clear schemas, crash-proof dispatch, timeouts, and parallelism — the plumbing every real agent needs.

Next → `04_react.ipynb`